<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_Fanar_SILMA_Fusion_Track1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Track 1: Fanar + SILMA Fusion (few-shot) — Idea 5

Two strong LLMs (Fanar-9B + SILMA-9B) with the same unified prompt, then combined
by voting.
Note: these are few-shot models (no training/epochs); we prompt them directly.
Idea: fusion helps when the two models are comparable in strength (Fanar 0.715,
SILMA expected close) — unlike fusing with a weak encoder, which failed earlier.

- Generates each model's predictions + a separate submission file
- Combines them with several strategies
- Saves to a new independent Drive folder `track1_fanar_silma` without overwriting
  any other file

Required in `data`: `test_seen.csv`. Requires a T4 GPU.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.5 MB/s eta 0:00:00


In [ ]:
import torch, os
assert torch.cuda.is_available(), "فعّل GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
DATA_DIR="./data"; OUT="./fanar_silma_outputs"; os.makedirs(OUT, exist_ok=True)
TEXT_COL="text"; TARGET_COL="target"

GPU: Tesla T4


In [ ]:
#prompts
SYSTEM=("أنت مصنّف مواقف عربي دقيق. مهمتك تحديد موقف كاتب التغريدة تجاه هدف معيّن. "
        "أجب بكلمة إنجليزية واحدة فقط دون أي شرح: Favor أو Against أو None. "
        "Favor إذا كان مؤيداً للهدف، Against إذا كان معارضاً، None إذا لم يظهر موقف واضح.")
FEW=[("التطعيم أنقذ ملايين الأرواح ولازم الكل ياخذه","لقاح كورونا","Favor"),
     ("ما أثق باللقاح وله أضرار كثيرة","لقاح كورونا","Against"),
     ("متى تفتح مراكز التطعيم؟","لقاح كورونا","None")]
def msgs(text,target):
    m=[{"role":"system","content":SYSTEM}]
    for tw,tg,l in FEW:
        m+=[{"role":"user","content":f"الهدف: {tg}\nالتغريدة: {tw}\nالموقف:"},{"role":"assistant","content":l}]
    m.append({"role":"user","content":f"الهدف: {target} (قيادة المرأة للسيارة)\nالتغريدة: {text}\nالموقف:"})
    return m
def parse(o):
    o=o.strip().lower()
    if "against" in o: return "Against"
    if "favor" in o or "support" in o: return "Favor"
    return "None"

In [ ]:
# Run a single LLM (few-shot) on the test set
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd, zipfile, gc
from tqdm.auto import tqdm
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

df=pd.read_csv(f"{DATA_DIR}/test_seen.csv", keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns: df=df.rename(columns={"tweet_text":TEXT_COL})
print("اختبار Track 1:", len(df))

def run_llm(model_id, tag):
    print(f"\n===== {tag} =====")
    tok=AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    mdl.eval(); preds=[]
    for _,r in tqdm(df.iterrows(), total=len(df), desc=tag):
        p=tok.apply_chat_template(msgs(str(r[TEXT_COL]),str(r[TARGET_COL])), add_generation_prompt=True, tokenize=False)
        e=tok(p, return_tensors="pt", return_token_type_ids=False).to(mdl.device)
        with torch.no_grad(): o=mdl.generate(**e, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
        preds.append(parse(tok.decode(o[0][e["input_ids"].shape[-1]:], skip_special_tokens=True)))
    print(f"توزيع {tag}:", pd.Series(preds).value_counts().to_dict())

    txt=f"{OUT}/submission_seen_{tag}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUT}/submission_seen_{tag}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    del mdl; gc.collect(); torch.cuda.empty_cache()
    return preds

fanar_pred = run_llm("QCRI/Fanar-1-9B-Instruct", "fanar")
silma_pred = run_llm("silma-ai/SILMA-9B-Instruct-v1.0", "silma")

اختبار Track 1: 352

===== fanar =====


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 18.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

silma:   0%|          | 0/352 [00:00<?, ?it/s]

توزيع silma: {'Favor': 201, 'Against': 147, 'None': 4}


In [ ]:
# Fusion by voting (two models -> need a rule on disagreement)
import pandas as pd
n=min(len(fanar_pred), len(silma_pred))
fanar_pred=fanar_pred[:n]; silma_pred=silma_pred[:n]

def save_sub(preds, name):
    import zipfile
    txt=f"{OUT}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUT}/{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    print(f"{name}: {pd.Series(preds).value_counts().to_dict()}")

agree=sum(1 for i in range(n) if fanar_pred[i]==silma_pred[i])
print(f"Models agreement: {agree}/{n} ({agree/n*100:.1f}%)")

# Strategy 1: on agreement take it; on disagreement prefer Fanar (historically stronger)
fuse_fanar=[fanar_pred[i] if fanar_pred[i]==silma_pred[i] else fanar_pred[i] for i in range(n)]
save_sub(fuse_fanar, "sub_fuse_fanar_priority")   # effectively Fanar; reference

# Strategy 2: on disagreement prefer SILMA
fuse_silma=[silma_pred[i] for i in range(n)]
save_sub(fuse_silma, "sub_silma_only")

# Strategy 3: on disagreement, if one is None and the other is not -> take the non-None (reduce abstention)
fuse_notnone=[]
for i in range(n):
    a,b=fanar_pred[i],silma_pred[i]
    if a==b: fuse_notnone.append(a)
    elif a=="None": fuse_notnone.append(b)
    elif b=="None": fuse_notnone.append(a)
    else: fuse_notnone.append(a)   # both non-None and differ -> prefer Fanar
save_sub(fuse_notnone, "sub_fuse_prefer_notnone")

print("\n>>> Upload: sub_silma_only + sub_fuse_prefer_notnone on Track 1 and compare with Fanar (0.7152) <<<")

اتفاق النموذجين: 257/352 (73.0%)
sub_fuse_fanar_priority: {'Against': 215, 'Favor': 115, 'None': 22}
sub_silma_only: {'Favor': 201, 'Against': 147, 'None': 4}
sub_fuse_prefer_notnone: {'Against': 218, 'Favor': 132, 'None': 2}

>>> ارفع: sub_silma_only + sub_fuse_prefer_notnone على Track 1 وقارن مع Fanar (0.7152) <<<
